In [ ]:
# Install Dependencies
!pip install -q transformers datasets peft bitsandbytes accelerate trl
!pip install -q openai scikit-learn pandas numpy matplotlib seaborn

In [ ]:
# Imports
import os
import json
import time
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from huggingface_hub import login

from openai import OpenAI
from datasets import load_dataset
from sklearn.metrics import classification_report, confusion_matrix

import torch
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
    TrainingArguments,
)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from trl import SFTTrainer

print("✓ All imports successful")
print(f"✓ PyTorch version: {torch.__version__}")
print(f"✓ GPU available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"✓ GPU: {torch.cuda.get_device_name(0)}")

In [ ]:
# Mount Drive + Constants & Config
from google.colab import userdata, drive

# Mount Google Drive
drive.mount("/content/drive")

# API Keys
OPENROUTER_API_KEY = userdata.get("OPENROUTER_API_KEY")
HF_TOKEN           = userdata.get("HF_TOKEN")

# OpenRouter
OPENROUTER_BASE_URL = "https://openrouter.ai/api/v1"
FRONTIER_MODEL      = "openai/gpt-4o-mini"

# Dataset
DATASET_NAME  = "jacob-hugging-face/job-descriptions"
DATASET_SPLIT = "train"
SAMPLE_SIZE   = 200

# Open-source model
OSS_MODEL_NAME = "meta-llama/Llama-3.1-8B-Instruct"

# Scoring tiers
SCORE_TIERS = {
    "Hot":  (70, 100),
    "Warm": (40, 69),
    "Cold": (0,  39),
}

# Paths — all persistent on Google Drive
DRIVE_BASE          = "/content/drive/MyDrive/saas_lead_scoring"
OUTPUT_DIR          = f"{DRIVE_BASE}/outputs"
FINE_TUNED_MODEL_DIR = f"{DRIVE_BASE}/fine_tuned_model"
RESULTS_FILE        = "frontier_results.json"

os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(FINE_TUNED_MODEL_DIR, exist_ok=True)

# Validate secrets
assert OPENROUTER_API_KEY, "Missing secret: OPENROUTER_API_KEY"
assert HF_TOKEN,           "Missing secret: HF_TOKEN"

print("✓ Google Drive mounted")
print("✓ Secrets loaded successfully")
print(f"✓ Frontier model  : {FRONTIER_MODEL}")
print(f"✓ OSS model       : {OSS_MODEL_NAME}")
print(f"✓ Outputs dir     : {OUTPUT_DIR}")
print(f"✓ Model dir       : {FINE_TUNED_MODEL_DIR}")

In [ ]:
#  Load & Explore Dataset
from huggingface_hub import login

login(token=HF_TOKEN, add_to_git_credential=False)

# Load dataset
dataset = load_dataset(DATASET_NAME, split=DATASET_SPLIT)
df = pd.DataFrame(dataset)

print(f"✓ Dataset loaded: {len(df)} records")
print(f"✓ Columns: {df.columns.tolist()}")
print(f"\n--- Sample Record ---")
print(df.iloc[0].to_dict())

In [ ]:
# Preprocess & Sample Data

# Inspect model_response to understand its structure
print("--- Sample model_response ---")
print(df["model_response"].iloc[0])
print("\n--- Null counts ---")
print(df.isnull().sum())
print(f"\n--- Description length stats ---")
print(df["description_length"].describe())

In [ ]:
# Preprocess & Sample Data

def parse_model_response(raw):
    try:
        return json.loads(raw)
    except (json.JSONDecodeError, TypeError):
        return None

# Parse model_response JSON
df["parsed_response"] = df["model_response"].apply(parse_model_response)

# Drop unparseable rows
df = df[df["parsed_response"].notnull()].reset_index(drop=True)

# Keep only relevant columns
df = df[["company_name", "position_title", "job_description", "parsed_response"]]

# Filter out very short descriptions (likely junk)
df = df[df["job_description"].str.len() >= 100].reset_index(drop=True)

# Sample for frontier model evaluation
df_sample = df.sample(n=min(SAMPLE_SIZE, len(df)), random_state=42).reset_index(drop=True)

print(f"✓ Clean dataset  : {len(df)} records")
print(f"✓ Sample size    : {len(df_sample)} records")
print(f"\n--- Sample Row ---")
print(f"Company   : {df_sample['company_name'].iloc[0]}")
print(f"Position  : {df_sample['position_title'].iloc[0]}")
print(f"JD length : {len(df_sample['job_description'].iloc[0])} chars")
print(f"Parsed    : {json.dumps(df_sample['parsed_response'].iloc[0], indent=2)}")

In [ ]:
#  Data Quality Filter (fixed)

def is_valid_response(parsed):
    if not parsed:
        return False
    valid_fields = []
    for v in parsed.values():
        # handle both string and list values
        if isinstance(v, list):
            text = " ".join(v).strip()
        elif isinstance(v, str):
            text = v.strip()
        else:
            continue
        if text.upper() not in ("N/A", "NONE", "NULL", ""):
            valid_fields.append(text)
    return len(valid_fields) >= 2

# Filter main dataset
df["is_valid"] = df["parsed_response"].apply(is_valid_response)
df_clean = df[df["is_valid"]].drop(columns=["is_valid"]).reset_index(drop=True)

# Resample from clean records
df_sample = df_clean.sample(n=min(SAMPLE_SIZE, len(df_clean)), random_state=42).reset_index(drop=True)

print(f"✓ Records after quality filter : {len(df_clean)}")
print(f"✓ Dropped                      : {len(df) - len(df_clean)} records")
print(f"✓ Final sample size            : {len(df_sample)}")
print(f"\n--- Clean Sample Row ---")
print(f"Company  : {df_sample['company_name'].iloc[0]}")
print(f"Position : {df_sample['position_title'].iloc[0]}")
print(f"Parsed   : {json.dumps(df_sample['parsed_response'].iloc[0], indent=2)}")

In [ ]:
#  Frontier Model Scoring Function

client = OpenAI(api_key=OPENROUTER_API_KEY, base_url=OPENROUTER_BASE_URL)

def build_prompt(row):
    parsed = row["parsed_response"]
    return f"""You are a B2B SaaS sales intelligence tool.

Analyze the job posting below and score how likely this company is to be
in-market for a B2B SaaS product RIGHT NOW, based on their hiring signals.

Company       : {row['company_name']}
Position      : {row['position_title']}
Responsibilities: {parsed.get('Core Responsibilities', 'N/A')}
Required Skills : {parsed.get('Required Skills', 'N/A')}
Experience Level: {parsed.get('Experience Level', 'N/A')}

Respond ONLY with a JSON object in this exact format:
{{
  "score": <integer 0-100>,
  "tier": "<Hot|Warm|Cold>",
  "reasoning": "<one sentence explanation>"
}}"""

def score_with_frontier(row):
    try:
        response = client.chat.completions.create(
            model=FRONTIER_MODEL,
            messages=[{"role": "user", "content": build_prompt(row)}],
            temperature=0.1,
        )
        raw = response.choices[0].message.content.strip()
        # strip markdown code fences if present
        raw = raw.replace("```json", "").replace("```", "").strip()
        result = json.loads(raw)
        return {
            "score":     int(result.get("score", 0)),
            "tier":      result.get("tier", "Cold"),
            "reasoning": result.get("reasoning", ""),
            "error":     None
        }
    except Exception as e:
        return {"score": 0, "tier": "Cold", "reasoning": "", "error": str(e)}

# Quick sanity check on one record
test_result = score_with_frontier(df_sample.iloc[0])
print(f"✓ Frontier model reachable")
print(f"  Company  : {df_sample.iloc[0]['company_name']}")
print(f"  Position : {df_sample.iloc[0]['position_title']}")
print(f"  Score    : {test_result['score']}")
print(f"  Tier     : {test_result['tier']}")
print(f"  Reasoning: {test_result['reasoning']}")
print(f"  Error    : {test_result['error']}")

In [ ]:
#  Run Frontier Model on Full Sample

results = []

for i, row in df_sample.iterrows():
    result = score_with_frontier(row)
    results.append({
        "company_name":   row["company_name"],
        "position_title": row["position_title"],
        "score":          result["score"],
        "tier":           result["tier"],
        "reasoning":      result["reasoning"],
        "error":          result["error"]
    })

    # Progress update every 20 records
    if (len(results)) % 20 == 0:
        print(f"  Processed {len(results)}/200...")

    time.sleep(0.5)  # avoid rate limiting

# Save results
df_results = pd.DataFrame(results)
df_results.to_json(os.path.join(OUTPUT_DIR, RESULTS_FILE), orient="records", indent=2)

# Summary
errors = df_results["error"].notnull().sum()
print(f"\n✓ Scoring complete")
print(f"  Total scored : {len(df_results)}")
print(f"  Errors       : {errors}")
print(f"\n--- Tier Distribution ---")
print(df_results["tier"].value_counts().to_string())
print(f"\n--- Score Stats ---")
print(df_results["score"].describe().round(2).to_string())

In [ ]:
#  Evaluate & Visualize Frontier Results

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
fig.suptitle("Frontier Model (GPT-4o-mini) — Scoring Results", fontsize=14, fontweight="bold")

# 1. Tier distribution bar chart
tier_counts = df_results["tier"].value_counts().reindex(["Hot", "Warm", "Cold"])
colors = ["#e74c3c", "#f39c12", "#3498db"]
axes[0].bar(tier_counts.index, tier_counts.values, color=colors, edgecolor="black", linewidth=0.5)
axes[0].set_title("Tier Distribution")
axes[0].set_xlabel("Tier")
axes[0].set_ylabel("Count")
for i, v in enumerate(tier_counts.values):
    axes[0].text(i, v + 1, str(v), ha="center", fontweight="bold")

# 2. Score distribution histogram
axes[1].hist(df_results["score"], bins=20, color="#2ecc71", edgecolor="black", linewidth=0.5)
axes[1].set_title("Score Distribution")
axes[1].set_xlabel("Score (0-100)")
axes[1].set_ylabel("Frequency")
axes[1].axvline(df_results["score"].mean(), color="red", linestyle="--", label=f'Mean: {df_results["score"].mean():.1f}')
axes[1].legend()

# 3. Score boxplot by tier
tier_order = ["Hot", "Warm", "Cold"]
tier_data = [df_results[df_results["tier"] == t]["score"].values for t in tier_order]
bp = axes[2].boxplot(tier_data, labels=tier_order, patch_artist=True)
for patch, color in zip(bp["boxes"], colors):
    patch.set_facecolor(color)
    patch.set_alpha(0.7)
axes[2].set_title("Score Range by Tier")
axes[2].set_xlabel("Tier")
axes[2].set_ylabel("Score")

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "frontier_results.png"), dpi=150, bbox_inches="tight")
plt.show()

print(f"\n--- Score by Tier ---")
print(df_results.groupby("tier")["score"].describe().round(2).to_string())
print(f"\n✓ Plot saved to {OUTPUT_DIR}/frontier_results.png")

In [ ]:
# Prepare Fine-Tuning Dataset

def build_training_example(row, result):
    prompt = build_prompt(row)
    completion = json.dumps({
        "score":     result["score"],
        "tier":      result["tier"],
        "reasoning": result["reasoning"]
    })
    # Llama instruct format
    return {
        "text": f"<|begin_of_text|><|start_header_id|>user<|end_header_id|>\n{prompt}<|eot_id|>"
                f"<|start_header_id|>assistant<|end_header_id|>\n{completion}<|eot_id|>"
    }

# Build training examples from all 200 frontier-scored records
training_data = [
    build_training_example(df_sample.iloc[i], results[i])
    for i in range(len(df_sample))
    if results[i]["error"] is None
]

# 80/20 train-eval split
split = int(len(training_data) * 0.8)
train_data = training_data[:split]
eval_data  = training_data[split:]

# Save to disk
train_path = os.path.join(OUTPUT_DIR, "train.json")
eval_path  = os.path.join(OUTPUT_DIR, "eval.json")

with open(train_path, "w") as f:
    json.dump(train_data, f, indent=2)

with open(eval_path, "w") as f:
    json.dump(eval_data, f, indent=2)

print(f"✓ Training examples : {len(train_data)}")
print(f"✓ Eval examples     : {len(eval_data)}")
print(f"\n--- Sample Training Record ---")
print(training_data[0]["text"][:500] + "...")

In [ ]:
login(token=HF_TOKEN, add_to_git_credential=False)

In [ ]:
#  Load Base Model (4-bit QLoRA)

# 4-bit quantization config
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
)

# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(
    OSS_MODEL_NAME,
    token=HF_TOKEN,
    trust_remote_code=True,
)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

# Load model
model = AutoModelForCausalLM.from_pretrained(
    OSS_MODEL_NAME,
    token=HF_TOKEN,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
)

print(f"✓ Model loaded     : {OSS_MODEL_NAME}")
print(f"✓ Quantization     : 4-bit NF4")
print(f"✓ Compute dtype    : bfloat16")
print(f"✓ Device map       : auto")

# GPU memory check
mem = torch.cuda.memory_allocated() / 1024**3
print(f"✓ GPU memory used  : {mem:.2f} GB")

In [ ]:
#  Prepare Model for QLoRA & Attach LoRA Adapters

# Prepare model for k-bit training
model = prepare_model_for_kbit_training(model)

# LoRA config
lora_config = LoraConfig(
    r=16,                     # rank — balance between capacity and memory
    lora_alpha=32,            # scaling factor
    target_modules=[          # attention layers to adapt
        "q_proj", "k_proj",
        "v_proj", "o_proj",
    ],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
)

# Attach LoRA adapters to model
model = get_peft_model(model, lora_config)

# Show trainable parameter count
trainable, total = model.get_nb_trainable_parameters()
print(f"✓ LoRA adapters attached")
print(f"  Trainable params : {trainable:,}")
print(f"  Total params     : {total:,}")
print(f"  Trainable %      : {100 * trainable / total:.4f}%")

In [ ]:
#  Load & Tokenize Training Data
from datasets import Dataset

# Load from saved JSON files
with open(train_path, "r") as f:
    train_records = json.load(f)

with open(eval_path, "r") as f:
    eval_records = json.load(f)

# Convert to HuggingFace Dataset
train_dataset = Dataset.from_list(train_records)
eval_dataset  = Dataset.from_list(eval_records)

# Tokenize
MAX_SEQ_LENGTH = 512

def tokenize(example):
    return tokenizer(
        example["text"],
        truncation=True,
        max_length=MAX_SEQ_LENGTH,
        padding="max_length",
    )

train_dataset = train_dataset.map(tokenize, batched=True)
eval_dataset  = eval_dataset.map(tokenize, batched=True)

print(f"✓ Train dataset : {len(train_dataset)} examples")
print(f"✓ Eval dataset  : {len(eval_dataset)} examples")
print(f"✓ Max seq length: {MAX_SEQ_LENGTH}")
print(f"\n--- Token length sample ---")
sample_len = len(train_dataset[0]["input_ids"])
print(f"  First example token length: {sample_len}")

In [ ]:
#  Configure Training Arguments & Train (fixed)

training_args = TrainingArguments(
    output_dir=FINE_TUNED_MODEL_DIR,
    num_train_epochs=3,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=4,
    warmup_steps=10,
    learning_rate=2e-4,
    bf16=True,
    logging_steps=10,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    report_to="none",
)

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    processing_class=tokenizer,
)

print("✓ Trainer configured")
print(f"  Epochs           : {training_args.num_train_epochs}")
print(f"  Batch size       : {training_args.per_device_train_batch_size}")
print(f"  Grad accumulation: {training_args.gradient_accumulation_steps}")
print(f"  Learning rate    : {training_args.learning_rate}")
print(f"\nStarting training...")

# Train
train_result = trainer.train()

print(f"\n✓ Training complete")
print(f"  Runtime         : {train_result.metrics['train_runtime']:.0f}s")
print(f"  Train loss      : {train_result.metrics['train_loss']:.4f}")

# Save final model
trainer.save_model(FINE_TUNED_MODEL_DIR)
tokenizer.save_pretrained(FINE_TUNED_MODEL_DIR)
print(f"✓ Model saved to  : {FINE_TUNED_MODEL_DIR}")

In [ ]:
#  Clear Memory, Load Fine-Tuned Model & Inference Function
import gc
from peft import PeftModel

# Clear GPU memory
gc.collect()
torch.cuda.empty_cache()
torch.cuda.synchronize()
print(f"✓ GPU cleared : {torch.cuda.memory_allocated() / 1024**3:.2f} GB allocated")

# Load base model
ft_model = AutoModelForCausalLM.from_pretrained(
    OSS_MODEL_NAME,
    token=HF_TOKEN,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
)

# Load LoRA adapters from Drive
ft_model = PeftModel.from_pretrained(ft_model, FINE_TUNED_MODEL_DIR)
ft_model.eval()
print(f"✓ Fine-tuned model loaded")
print(f"  GPU memory allocated : {torch.cuda.memory_allocated() / 1024**3:.2f} GB")

def extract_json(text):
    match = re.search(r'\{.*?\}', text, re.DOTALL)
    if match:
        return match.group(0)
    return None

def score_with_finetuned(row):
    prompt = build_prompt(row)
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=512).to("cuda")
    with torch.no_grad():
        outputs = ft_model.generate(
            **inputs,
            max_new_tokens=128,
            temperature=0.1,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id,
        )
    generated = tokenizer.decode(outputs[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
    try:
        clean    = generated.replace("```json", "").replace("```", "").strip()
        json_str = extract_json(clean)
        if not json_str:
            raise ValueError("No JSON object found in output")
        result = json.loads(json_str)
        return {
            "score":     int(result.get("score", 0)),
            "tier":      result.get("tier", "Cold"),
            "reasoning": result.get("reasoning", ""),
            "error":     None
        }
    except Exception as e:
        return {"score": 0, "tier": "Cold", "reasoning": "", "error": str(e)}

# Sanity check
test_ft = score_with_finetuned(df_sample.iloc[0])
print(f"\n✓ Inference working")
print(f"  Company  : {df_sample.iloc[0]['company_name']}")
print(f"  Position : {df_sample.iloc[0]['position_title']}")
print(f"  Score    : {test_ft['score']}")
print(f"  Tier     : {test_ft['tier']}")
print(f"  Reasoning: {test_ft['reasoning']}")
print(f"  Error    : {test_ft['error']}")

In [ ]:
#  Re-run Full Sample with Fixed Inference

BATCH_SIZE = 40
ft_results = []
batch_num  = 0

for start in range(0, len(df_sample), BATCH_SIZE):
    batch     = df_sample.iloc[start:start + BATCH_SIZE]
    batch_num += 1
    print(f"\n  Processing batch {batch_num} ({start+1}-{min(start+BATCH_SIZE, len(df_sample))})...")

    for i, row in batch.iterrows():
        result = score_with_finetuned(row)
        ft_results.append({
            "company_name":   row["company_name"],
            "position_title": row["position_title"],
            "score":          result["score"],
            "tier":           result["tier"],
            "reasoning":      result["reasoning"],
            "error":          result["error"]
        })

    # Save after every batch
    df_ft_results = pd.DataFrame(ft_results)
    df_ft_results.to_json(
        os.path.join(OUTPUT_DIR, "finetuned_results.json"),
        orient="records", indent=2
    )
    print(f"  ✓ Batch {batch_num} saved — {len(ft_results)} total so far")

errors = df_ft_results["error"].notnull().sum()
print(f"\n✓ Scoring complete")
print(f"  Total scored : {len(df_ft_results)}")
print(f"  Errors       : {errors}")
print(f"\n--- Tier Distribution ---")
print(df_ft_results["tier"].value_counts().to_string())
print(f"\n--- Score Stats ---")
print(df_ft_results["score"].describe().round(2).to_string())

In [ ]:
#  Final Comparison & Evaluation

# Load both result sets
df_frontier  = pd.read_json(os.path.join(OUTPUT_DIR, RESULTS_FILE))
df_finetuned = pd.read_json(os.path.join(OUTPUT_DIR, "finetuned_results.json"))

# Align on same 100 records
df_frontier  = df_frontier.iloc[:100].reset_index(drop=True)
df_finetuned = df_finetuned.iloc[:100].reset_index(drop=True)

# ── 1. Tier Agreement ────────────────────────────────────────────────
agreement      = (df_frontier["tier"] == df_finetuned["tier"]).sum()
agreement_rate = agreement / len(df_frontier) * 100

# ── 2. Score Correlation ─────────────────────────────────────────────
correlation = df_frontier["score"].corr(df_finetuned["score"])

# ── 3. Score Diversity (std) ─────────────────────────────────────────
frontier_std  = df_frontier["score"].std()
finetuned_std = df_finetuned["score"].std()

# ── 4. Print Summary ─────────────────────────────────────────────────
print("=" * 55)
print("        MODEL COMPARISON REPORT")
print("=" * 55)
print(f"  Records evaluated     : {len(df_frontier)}")
print(f"  Tier agreement rate   : {agreement_rate:.1f}%")
print(f"  Score correlation     : {correlation:.3f}")
print()
print(f"  {'Metric':<25} {'Frontier':>10} {'Fine-tuned':>10}")
print(f"  {'-'*45}")
print(f"  {'Mean score':<25} {df_frontier['score'].mean():>10.1f} {df_finetuned['score'].mean():>10.1f}")
print(f"  {'Std deviation':<25} {frontier_std:>10.2f} {finetuned_std:>10.2f}")
print(f"  {'Min score':<25} {df_frontier['score'].min():>10.1f} {df_finetuned['score'].min():>10.1f}")
print(f"  {'Max score':<25} {df_frontier['score'].max():>10.1f} {df_finetuned['score'].max():>10.1f}")
print()
print(f"  {'Tier':<10} {'Frontier':>10} {'Fine-tuned':>10}")
print(f"  {'-'*30}")
for tier in ["Hot", "Warm", "Cold"]:
    f_count  = (df_frontier["tier"]  == tier).sum()
    ft_count = (df_finetuned["tier"] == tier).sum()
    print(f"  {tier:<10} {f_count:>10} {ft_count:>10}")
print("=" * 55)

# ── 5. Visualize ─────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
fig.suptitle("Frontier (GPT-4o-mini) vs Fine-Tuned (Llama 3.1 8B)", fontsize=14, fontweight="bold")

# Tier distribution comparison
tier_order   = ["Hot", "Warm", "Cold"]
frontier_counts  = [( df_frontier["tier"] == t).sum() for t in tier_order]
finetuned_counts = [(df_finetuned["tier"] == t).sum() for t in tier_order]
x = range(len(tier_order))
axes[0].bar([i - 0.2 for i in x], frontier_counts,  width=0.4, label="Frontier",  color="#3498db", edgecolor="black", linewidth=0.5)
axes[0].bar([i + 0.2 for i in x], finetuned_counts, width=0.4, label="Fine-tuned", color="#e74c3c", edgecolor="black", linewidth=0.5)
axes[0].set_xticks(x)
axes[0].set_xticklabels(tier_order)
axes[0].set_title("Tier Distribution")
axes[0].set_ylabel("Count")
axes[0].legend()

# Score distribution overlay
axes[1].hist(df_frontier["score"],  bins=20, alpha=0.6, label="Frontier",  color="#3498db", edgecolor="black", linewidth=0.5)
axes[1].hist(df_finetuned["score"], bins=20, alpha=0.6, label="Fine-tuned", color="#e74c3c", edgecolor="black", linewidth=0.5)
axes[1].set_title("Score Distribution")
axes[1].set_xlabel("Score (0-100)")
axes[1].set_ylabel("Frequency")
axes[1].legend()

# Score scatter plot
axes[2].scatter(df_frontier["score"], df_finetuned["score"], alpha=0.5, color="#2ecc71", edgecolor="black", linewidth=0.3)
axes[2].plot([0, 100], [0, 100], "r--", linewidth=1, label="Perfect agreement")
axes[2].set_title(f"Score Correlation (r={correlation:.2f})")
axes[2].set_xlabel("Frontier Score")
axes[2].set_ylabel("Fine-tuned Score")
axes[2].legend()

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "model_comparison.png"), dpi=150, bbox_inches="tight")
plt.show()

# ── 6. Verdict ───────────────────────────────────────────────────────
print("\n── VERDICT ──────────────────────────────────────────")
if agreement_rate >= 70:
    print(f"  ✅ Fine-tuned model MATCHES frontier tier accuracy")
    print(f"     ({agreement_rate:.1f}% tier agreement)")
else:
    print(f"    Fine-tuned model PARTIALLY matches frontier")
    print(f"     ({agreement_rate:.1f}% tier agreement — target: 70%+)")

if finetuned_std > frontier_std:
    print(f"   Fine-tuned model produces MORE score diversity")
    print(f"     (std {finetuned_std:.2f} vs frontier {frontier_std:.2f})")
else:
    print(f"    Frontier model produces more score diversity")
    print(f"     (std {frontier_std:.2f} vs fine-tuned {finetuned_std:.2f})")

print(f"   Score correlation : {correlation:.3f}")
print(f"   Training params   : 13.6M / 8B total (0.17%)")
print(f"   Inference cost    : $0 (local) vs API cost (frontier)")
print("─" * 55)